In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install gguf protobuf

In [ ]:
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
dtype = None
load_in_4bit = True

fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit",
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",
]
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,

)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):

        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass
from datasets import load_dataset, DatasetDict
from datasets import load_dataset
data_files = [
    '/content/drive/MyDrive/total_examples.json',
    '/content/drive/MyDrive/total_examples2.json'
]

dataset_1 = load_dataset('json', data_files={'train': data_files[0]}, split='train')
dataset_2 = load_dataset('json', data_files={'train': data_files[1]}, split='train')

dataset_1 = dataset_1.map(formatting_prompts_func, batched=True)
dataset_2 = dataset_2.map(formatting_prompts_func, batched=True)

train_1, val_1 = dataset_1.train_test_split(test_size=0.1, seed=3407).values()
train_2, val_2 = dataset_2.train_test_split(test_size=0.1, seed=3407).values()

train_dataset = train_2
val_dataset = val_2

In [ ]:
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

print(dataset_dict)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

args_for_Data2 = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=3,
    num_train_epochs=10,
    learning_rate=1e-3,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_steps=1,
    load_best_model_at_end=True,
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset =val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=args_for_Data2,
)


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

In [ ]:
eval_loss = eval_results["eval_loss"]

eval_loss_threshold = 0.7
max_additional_epochs = 100
additional_epochs_trained = 0

while eval_loss > eval_loss_threshold and additional_epochs_trained < max_additional_epochs:
    print(f"评估损失 {eval_loss} 高于阈值 {eval_loss_threshold}，继续训练额外的 epoch...")

    trainer.args.num_train_epochs += 10
    additional_epochs_trained += 1

    trainer.train(resume_from_checkpoint=True)

    eval_results = trainer.evaluate()
    eval_loss = eval_results["eval_loss"]

if eval_loss <= eval_loss_threshold:
    print(f"评估损失 {eval_loss} 已低于阈值 {eval_loss_threshold}，训练完成。")
else:
    print(f"已达到最大额外训练次数，但评估损失仍高于阈值。")

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")